# 3x Leveraged ETF v1 — DSL Engine Replication

Replicates `triple_leveraged_etf_v1.ipynb` using the QuantDSL backtest engine.

**Strategy**: 50% TQQQ / 50% TMF, per-asset SMA-200 trend filter with 5% hysteresis re-entry.  
Each leg independently parks in IEF when its underlying (QQQ / TLT) drops below its SMA-200.

**DSL signal mapping**:  
`ExternalFactor(per_instrument=True)` → one float column per ticker → `GreaterEqual(..., 0.5)` → `MaskSelector`

**Known difference vs v1**: DSL engine currently rebalances *daily*; v1 rebalances every 2 months.  
Signal logic (QQQ/TLT SMA-200 with 5% hysteresis) is bit-exact.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import warnings, subprocess as _sp, sys, importlib
from pathlib import Path

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path(_sp.check_output(
    "git rev-parse --show-toplevel", shell=True, text=True, cwd="."
).strip())
sys.path.insert(0, str(PROJECT_ROOT / 'signum'))
import signum.engine.chart, signum.engine.dashboard, signum
importlib.reload(signum.engine.chart); importlib.reload(signum.engine.dashboard); importlib.reload(signum)
from signum import Chart, Dashboard

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Leveraged ETF prices (TQQQ, TMF, IEF)
df = pd.read_parquet(PROJECT_ROOT / 'equities' / 'triple_leveraged_etfs.parquet')
prices = df.pivot(index='date', columns='ticker', values='close').sort_index()
prices = prices[['TQQQ', 'TMF', 'IEF']].dropna()

# Open prices for next-day execution (matches v1 T+1 open fill)
open_prices = df.pivot(index='date', columns='ticker', values='open').sort_index()
open_prices = open_prices[['TQQQ', 'TMF', 'IEF']].reindex(prices.index).ffill()

print(f"Leveraged ETFs: {prices.index[0].date()} → {prices.index[-1].date()} ({len(prices)} days)")

# Download underlying QQQ & TLT for SMA signals (400-day warmup for SMA-200)
start = prices.index.min() - pd.Timedelta(days=400)
end   = prices.index.max()
qqq = yf.download('QQQ', start=start, end=end, progress=False)['Close'].squeeze()
tlt = yf.download('TLT', start=start, end=end, progress=False)['Close'].squeeze()
qqq = qqq.reindex(prices.index, method='ffill')
tlt = tlt.reindex(prices.index, method='ffill')
print(f"QQQ: {len(qqq)} rows  |  TLT: {len(tlt)} rows")

In [ ]:
# ── Strategy Parameters (identical to v1) ────────────────────────────────────
INITIAL_CAPITAL  = 100_000
TARGET_WEIGHT    = 0.50
REBALANCE_MONTHS = 2
SLIPPAGE_PCT     = 0.0025   # 0.25% flat per trade
SMA_PERIOD       = 200
EXIT_BUFFER      = 0.00     # exit when underlying < SMA
ENTER_BUFFER     = 0.05     # re-enter when underlying > SMA * 1.05

# ── SMA-200 Signals with Hysteresis (exact copy from v1) ─────────────────────
qqq_sma = qqq.rolling(SMA_PERIOD).mean()
tlt_sma = tlt.rolling(SMA_PERIOD).mean()

tqqq_hold = []; tmf_hold = []
tq_state = True; tm_state = True

for dt in prices.index:
    q, t   = qqq.get(dt, np.nan), tlt.get(dt, np.nan)
    qs, ts = qqq_sma.get(dt, np.nan), tlt_sma.get(dt, np.nan)
    if pd.isna(qs) or pd.isna(q):
        tqqq_hold.append(True); tmf_hold.append(True); continue
    if     tq_state and q < qs * (1 + EXIT_BUFFER):    tq_state = False
    elif not tq_state and q > qs * (1 + ENTER_BUFFER): tq_state = True
    if     tm_state and t < ts * (1 + EXIT_BUFFER):    tm_state = False
    elif not tm_state and t > ts * (1 + ENTER_BUFFER): tm_state = True
    tqqq_hold.append(tq_state); tmf_hold.append(tm_state)

tqqq_signal = pd.Series(tqqq_hold, index=prices.index, name='tqqq_hold')
tmf_signal  = pd.Series(tmf_hold,  index=prices.index, name='tmf_hold')
print(f"TQQQ in IEF: {(~tqqq_signal).sum()} days ({(~tqqq_signal).mean()*100:.1f}%)  |  "
      f"TMF in IEF: {(~tmf_signal).sum()} days ({(~tmf_signal).mean()*100:.1f}%)")

# ── v1 Strategy Engine (exact copy from v1 notebook) ─────────────────────────
def run_strategy(prices, open_prices, tqqq_signal, tmf_signal,
                 rebal_months=REBALANCE_MONTHS, slippage=SLIPPAGE_PCT,
                 initial_capital=INITIAL_CAPITAL):
    """Signal T close → trade fills T+1 open. Mark-to-market at close."""
    dates = prices.index
    half  = initial_capital / 2
    tq_sh = tq_ief = tm_sh = tm_ief = 0.0
    tq_in = tm_in = False
    month_ends      = dates.to_series().groupby(dates.to_period('M')).last()
    rebal_dates_set = set(month_ends.iloc[::rebal_months].values)
    last_rebal = None; initialized = False
    pending_tq = pending_tm = None; pending_rebal = False
    records = []; trade_log = []

    for i, dt in enumerate(dates):
        tp_c = prices.loc[dt,'TQQQ']; mp_c = prices.loc[dt,'TMF']; ip_c = prices.loc[dt,'IEF']
        tp_o = open_prices.loc[dt,'TQQQ']; mp_o = open_prices.loc[dt,'TMF']; ip_o = open_prices.loc[dt,'IEF']

        if pd.isna(tqqq_signal.get(dt)) or pd.isna(tmf_signal.get(dt)):
            records.append({'date':dt,'tqqq_leg':half,'tmf_leg':half,'portfolio_value':initial_capital,
                            'tqqq_in':True,'tmf_in':True,'weight_tqqq':0.5,'weight_tmf':0.5}); continue

        if not initialized:
            if tqqq_signal.loc[dt]: tq_sh = half/tp_o; tq_in = True
            else:                   tq_ief = half/ip_o; tq_in = False
            if tmf_signal.loc[dt]:  tm_sh = half/mp_o; tm_in = True
            else:                   tm_ief = half/ip_o; tm_in = False
            last_rebal = dt.to_period('M'); initialized = True
            tq_v = tq_sh*tp_c+tq_ief*ip_c; tm_v = tm_sh*mp_c+tm_ief*ip_c
            records.append({'date':dt,'tqqq_leg':tq_v,'tmf_leg':tm_v,'portfolio_value':tq_v+tm_v,
                            'tqqq_in':tq_in,'tmf_in':tm_in,'weight_tqqq':tq_v/(tq_v+tm_v),'weight_tmf':tm_v/(tq_v+tm_v)}); continue

        if pending_tq == 'exit':
            v = tq_sh*tp_o*(1-slippage); tq_sh=0; tq_ief=v/ip_o; tq_in=False
            trade_log.append((dt,'TQQQ','EXIT→IEF',v))
        elif pending_tq == 'enter':
            v = tq_ief*ip_o*(1-slippage); tq_ief=0; tq_sh=v/tp_o; tq_in=True
            trade_log.append((dt,'TQQQ','IEF→ENTER',v))
        pending_tq = None

        if pending_tm == 'exit':
            v = tm_sh*mp_o*(1-slippage); tm_sh=0; tm_ief=v/ip_o; tm_in=False
            trade_log.append((dt,'TMF','EXIT→IEF',v))
        elif pending_tm == 'enter':
            v = tm_ief*ip_o*(1-slippage); tm_ief=0; tm_sh=v/mp_o; tm_in=True
            trade_log.append((dt,'TMF','IEF→ENTER',v))
        pending_tm = None

        if pending_rebal:
            tq_r=tq_sh*tp_o+tq_ief*ip_o; tm_r=tm_sh*mp_o+tm_ief*ip_o; tot=tq_r+tm_r
            cost=abs(tq_r-tot/2)*slippage; tot-=cost; tgt=tot/2
            if tq_in: tq_sh=tgt/tp_o; tq_ief=0
            else:     tq_ief=tgt/ip_o; tq_sh=0
            if tm_in: tm_sh=tgt/mp_o; tm_ief=0
            else:     tm_ief=tgt/ip_o; tm_sh=0
            last_rebal=dt.to_period('M'); trade_log.append((dt,'REBAL','equalize',tgt)); pending_rebal=False

        tq_v=tq_sh*tp_c+tq_ief*ip_c; tm_v=tm_sh*mp_c+tm_ief*ip_c
        is_rebal = (dt in rebal_dates_set and last_rebal is not None
                    and dt.to_period('M') != last_rebal)
        if     tq_in and not tqqq_signal.loc[dt]: pending_tq='exit'
        elif not tq_in and tqqq_signal.loc[dt]:   pending_tq='enter'
        if     tm_in and not tmf_signal.loc[dt]:  pending_tm='exit'
        elif not tm_in and tmf_signal.loc[dt]:    pending_tm='enter'
        if is_rebal: pending_rebal=True

        pv = tq_v+tm_v
        records.append({'date':dt,'tqqq_leg':tq_v,'tmf_leg':tm_v,'portfolio_value':pv,
                        'tqqq_in':tq_in,'tmf_in':tm_in,
                        'weight_tqqq':tq_v/pv if pv>0 else 0,'weight_tmf':tm_v/pv if pv>0 else 0})

    return pd.DataFrame(records).set_index('date'), pd.DataFrame(trade_log, columns=['date','asset','action','value'])


def calculate_metrics(equity, risk_free_rate=0.02):
    r     = equity.pct_change().dropna()
    years = (equity.index[-1] - equity.index[0]).days / 365.25
    cagr  = (equity.iloc[-1] / equity.iloc[0]) ** (1/years) - 1
    vol   = r.std() * np.sqrt(252)
    dn    = r[r<0].std() * np.sqrt(252)
    dd    = (equity - equity.cummax()) / equity.cummax()
    return {
        'Final Value':  f"${equity.iloc[-1]:,.0f}",
        'CAGR':         f"{cagr*100:.1f}%",
        'Volatility':   f"{vol*100:.1f}%",
        'Sharpe':       f"{(cagr-risk_free_rate)/vol:.2f}" if vol>0 else '-',
        'Sortino':      f"{(cagr-risk_free_rate)/dn:.2f}"  if dn>0  else '-',
        'Max Drawdown': f"{dd.min()*100:.1f}%",
    }


# Run v1
v1_results, v1_trades = run_strategy(prices, open_prices, tqqq_signal, tmf_signal)
v1_equity  = v1_results['portfolio_value']
v1_metrics = calculate_metrics(v1_equity)
targets    = {'CAGR': '23.8%', 'Sharpe': '0.95', 'Max Drawdown': '-38.7%', 'Final Value': '$2,711,812'}

print(f"{'Metric':<16} {'v1':>14} {'Article':>14}")
print("-" * 44)
for k in ['Final Value', 'CAGR', 'Volatility', 'Sharpe', 'Sortino', 'Max Drawdown']:
    print(f"{k:<16} {v1_metrics[k]:>14} {targets.get(k, '-'):>14}")

---
## DSL Replication

Save per-instrument boolean signals to parquet → load via `ExternalFactor(per_instrument=True)` → run QuantDSL engine.

**Signal routing (IEF logic ensures always 2 selected → EqualWeight = 50/50):**

| Instrument | Signal column | When eligible |
|---|---|---|
| TQQQ | `tqqq_signal` | QQQ above SMA-200 (5% hysteresis re-entry) |
| TMF  | `tmf_signal`  | TLT above SMA-200 (5% hysteresis re-entry) |
| IEF  | `NOT(TQQQ AND TMF)` | Whenever ≥ 1 leg is filtered out |

In [ ]:
# Save per-instrument SMA signals as float parquet for ExternalFactor(per_instrument=True)
# IEF = True whenever ≥1 leg is filtered → always 2 assets selected → EqualWeight = 50/50
ief_eligible    = ~(tqqq_signal & tmf_signal)
sma_eligible_df = pd.DataFrame({
    'TQQQ': tqqq_signal.astype(float),
    'TMF':  tmf_signal.astype(float),
    'IEF':  ief_eligible.astype(float),
})

_sig_path = PROJECT_ROOT / 'data' / 'signals' / 'triple_lev_sma_eligible.parquet'
_sig_path.parent.mkdir(parents=True, exist_ok=True)
sma_eligible_df.to_parquet(_sig_path)

print(f"Saved → {_sig_path.relative_to(PROJECT_ROOT)}")
print(f"\nAllocation states (daily counts):")
print(sma_eligible_df.astype(bool).value_counts().to_string())

In [ ]:
import os, sys as _sys
_src = str(PROJECT_ROOT / 'src')
if _src not in _sys.path:
    _sys.path.insert(0, _src)
os.chdir(str(PROJECT_ROOT))

from quantdsl_backtest.dsl.strategy import Strategy
from quantdsl_backtest.dsl.data_config import DataConfig
from quantdsl_backtest.dsl.universe import Universe, HasHistory
from quantdsl_backtest.dsl.factors import ExternalFactor
from quantdsl_backtest.dsl.signals import MaskFromBoolean, GreaterEqual
from quantdsl_backtest.dsl.portfolio import (
    LongShortPortfolio, Book, MaskSelector, BottomN, EqualWeight,
)
from quantdsl_backtest.dsl.execution import (
    Execution, OrderPolicy, LatencyModel, PowerLawSlippageModel, VolumeParticipation,
)
from quantdsl_backtest.dsl.costs import Costs, Commission, BorrowCost, FinancingCost, StaticFees
from quantdsl_backtest.dsl.backtest_config import BacktestConfig, Reporting, RiskChecks
from quantdsl_backtest.engine.analytics.types import StrategyAnalyticsConfig


def build_strategy() -> Strategy:
    """
    v1 strategy expressed in QuantDSL.

    Signal mapping:
      ExternalFactor(per_instrument=True) loads the pre-computed wide parquet
      (columns: TQQQ, TMF, IEF with float 0.0 / 1.0 values).
      GreaterEqual(..., 0.5) converts to boolean mask.
      MaskSelector selects all instruments where mask=True.
      EqualWeight → always 50/50 (IEF eligibility guarantees exactly 2 instruments).

    Execution: signal_delay_bars=1 matches v1's T+1 execution (signal on T close,
    weights applied at T+1). Slippage: power-law ~25 bps base ≈ v1's flat 0.25%.
    """
    data_cfg = DataConfig(
        source="parquet://equities/triple_leveraged_etfs.parquet",
        calendar="XNYS", frequency="1d",
        start=str(prices.index.min().date()),
        end=str(prices.index.max().date()),
        price_adjustment="split_dividend",
        fields=["open", "high", "low", "close", "volume"],
    )

    universe = Universe(
        name="TripleLevSMAFilter", id_field="ticker",
        static_instruments=["TQQQ", "TMF", "IEF"],
        filters=[HasHistory(min_days=5)],
    )

    factors = {
        "sma_eligible_raw": ExternalFactor(
            name="sma_eligible_raw",
            path="data/signals/triple_lev_sma_eligible.parquet",
            per_instrument=True,   # columns: TQQQ, TMF, IEF (float 0.0 / 1.0)
        ),
    }

    signals = {
        "sma_eligible": MaskFromBoolean(
            name="sma_eligible",
            expr=GreaterEqual(left="sma_eligible_raw", right=0.5),
        ),
    }

    # MaskSelector → selects all instruments where sma_eligible=True
    # IEF eligible = NOT(TQQQ AND TMF) → always exactly 2 instruments selected
    # EqualWeight on 2 selected → 50% / 50%  (or 100% IEF when both signals are off)
    portfolio = LongShortPortfolio(
        long_book=Book(
            name="long_book",
            selector=MaskSelector(signal_name="sma_eligible"),
            weighting=EqualWeight(),
        ),
        short_book=Book(
            name="short_book",
            selector=BottomN(factor_name="sma_eligible_raw", n=0),  # empty
            weighting=EqualWeight(),
        ),
        rebalance_frequency="1d",
        rebalance_at="market_close",
        signal_delay_bars=1,          # signal at T close → weights at T+1 (matches v1)
        target_gross_leverage=1.0,
        target_net_exposure=1.0,
        max_abs_weight_per_name=1.0,  # allow 100% IEF
    )

    execution = Execution(
        order_policy=OrderPolicy(default_order_type="MOC", time_in_force="DAY"),
        latency=LatencyModel(signal_to_order_delay_bars=0, market_latency_ms=0),
        slippage=PowerLawSlippageModel(base_bps=25.0, k=10.0, exponent=0.5),
        volume_limits=VolumeParticipation(max_participation=1.0, mode="proportional"),
    )

    costs = Costs(
        commission=Commission(type="bps_notional", amount=0.0),
        borrow=BorrowCost(default_annual_rate=0.0),
        financing=FinancingCost(base_rate_curve="SOFR", spread_bps=0),
        fees=StaticFees(nav_fee_annual=0.0),
    )

    backtest_cfg = BacktestConfig(
        engine="event_driven",
        cash_initial=float(INITIAL_CAPITAL),
        risk_checks=RiskChecks(),
        reporting=Reporting(strategyAnalytics=StrategyAnalyticsConfig(
            output_dir="outputs/triple_leveraged_etf",
            title="3x Leveraged ETF v1 (QuantDSL)")),
    )

    return Strategy(
        name="triple_lev_sma_filter_dsl",
        data=data_cfg, universe=universe, factors=factors,
        signals=signals, portfolio=portfolio,
        execution=execution, costs=costs, backtest=backtest_cfg,
    )


strategy = build_strategy()
print(f"Strategy: {strategy.name}")
print(f"  Universe: {strategy.universe.static_instruments}")
print(f"  Engine:   {strategy.backtest.engine}  |  signal_delay_bars={strategy.portfolio.signal_delay_bars}")

In [ ]:
from quantdsl_backtest.engine.backtest_runner import run_backtest

dsl_result = run_backtest(strategy)
dsl_equity  = dsl_result.equity
dsl_metrics = calculate_metrics(dsl_equity)

print(f"DSL: {dsl_equity.index.min().date()} → {dsl_equity.index.max().date()}  ({len(dsl_equity)} bars)")
print(f"  Final: {dsl_metrics['Final Value']}  CAGR: {dsl_metrics['CAGR']}  "
      f"Sharpe: {dsl_metrics['Sharpe']}  Max DD: {dsl_metrics['Max Drawdown']}")

## Results: v1 vs DSL Engine

In [ ]:
# Align to shared date range
combined  = pd.concat([v1_equity.rename('v1'), dsl_equity.rename('DSL')], axis=1).dropna()
v1_m_cmp  = calculate_metrics(combined['v1'])
dsl_m_cmp = calculate_metrics(combined['DSL'])

print(f"{'Metric':<16} {'v1':>14} {'DSL Engine':>14} {'Article':>14}")
print("-" * 62)
for k in ['Final Value', 'CAGR', 'Volatility', 'Sharpe', 'Sortino', 'Max Drawdown']:
    print(f"{k:<16} {v1_m_cmp.get(k,'-'):>14} {dsl_m_cmp.get(k,'-'):>14} {targets.get(k,'-'):>14}")

print()
print("Differences:")
print("  Rebalance: v1 = bimonthly  |  DSL = daily (engine limitation)")
print("  Slippage:  v1 = flat 0.25%  |  DSL = power-law 25 bps base")

In [ ]:
# Equity curve comparison
any_ief_pos = pd.DataFrame({
    'position': ((~v1_results['tqqq_in']) | (~v1_results['tmf_in'])).astype(int)
}, index=v1_results.index).reindex(combined.index)

Dashboard(
    panes=[
        (Chart(theme='midnight', height=320, y_format='kmb')
         .line(combined['v1'],  name='v1 (bimonthly rebal)', color='#2196F3', width=2)
         .line(combined['DSL'], name='DSL (daily rebal)',    color='#9C27B0', width=2)
         .shade(any_ief_pos, position_col='position', color='#FF5722', opacity=0.10)
         .stats_legend({
             'v1 CAGR':     v1_m_cmp['CAGR'],
             'v1 Sharpe':   v1_m_cmp['Sharpe'],
             'DSL CAGR':    dsl_m_cmp['CAGR'],
             'DSL Sharpe':  dsl_m_cmp['Sharpe'],
         }, position='top-left')),
        (Chart(theme='midnight', height=140)
         .baseline((combined['DSL'] - combined['v1']).dropna(),
                   base_value=0, topLineColor='#9C27B0', bottomLineColor='#2196F3')),
    ],
    titles=['Equity Curve: v1 vs DSL (orange = filter active)',
            'DSL − v1 Difference ($)'],
    theme='midnight',
).show()

---
## Summary

### Benefits of DSL over manual Python

| Aspect | Manual (v1) | DSL Engine |
|--------|------------|------------|
| Strategy spec | Logic, execution & accounting interleaved | Declarative dataclasses, separated from engine |
| Costs / slippage | Hand-coded per strategy | Engine applies universally (power-law, borrow, financing) |
| Trade log | Not tracked | Full `result.trades` DataFrame |
| Tearsheet | Manual plots | Auto HTML report (`outputs/triple_leveraged_etf/`) |
| Engine swap | Full rewrite | `BacktestConfig(engine="vectorized")` |
| Reproducibility | Fragile notebook state | Git-clean config, rerunnable |

### DSL signal design (this notebook)
- **`ExternalFactor(per_instrument=True)`** — new feature: loads a wide parquet where each column is a ticker; routes values to the matching instrument (not broadcast). Enables per-asset signals like QQQ SMA → TQQQ, TLT SMA → TMF.
- **`MaskSelector`** — selects instruments by boolean signal (no ranking needed). Combined with `IEF = NOT(TQQQ AND TMF)`, always 2 instruments are selected → `EqualWeight` = exact 50/50.

### Known differences vs v1
| | v1 | DSL |
|---|---|---|
| Rebalance | Every 2 months | Daily (engine limitation) |
| Slippage | Flat 0.25% | Power-law 25 bps base |
| Execution | T+1 open | T+1 (signal_delay_bars=1) |
| Signal logic | Identical | Identical |